Boiler plate code

In [1]:
import langchain 
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.prompts.chat import ChatPromptTemplate

load_dotenv()

google_api_key = os.getenv("GOOGLE_API_KEY")
google_llm = ChatGoogleGenerativeAI(
    #temperature = 0, 
    model ="gemini-3.5-flash-lite", 
    api_key = google_api_key,
    max_output_tokens =60
    )

LCEL - Single Session

In [9]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.prompts import MessagesPlaceholder

prompt_template = ChatPromptTemplate.from_messages([
    ("system","You are a helpful assistant . Answer the questions very shortly and briefly"),
    MessagesPlaceholder("history"),
    ("human","{input}")
])

chain = prompt_template | google_llm | StrOutputParser()

conversation =[]

def chat_with_bot(payload):
    input = payload["input"]
    conversation.append(HumanMessage(content=input))
    res = chain.invoke({"input": input, "history": conversation})
    conversation.append(AIMessage(content=res))
    return res

print(conversation)
res1=chat_with_bot({"input":"Who is Joseph vijay?"})
print(conversation)
res2=chat_with_bot({"input":"What is his age?"})

print(res1,"\n",res2)

[]
[HumanMessage(content='Who is Joseph vijay?', additional_kwargs={}, response_metadata={}), AIMessage(content='Joseph Vijay, commonly known as **Vijay**, is a famous Indian actor and politician who predominantly works in Tamil cinema.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]
Joseph Vijay, commonly known as **Vijay**, is a famous Indian actor and politician who predominantly works in Tamil cinema. 
 He is 50 years old (born June 22, 1974).


LCEL - Multi Session

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.prompts import MessagesPlaceholder
from langchain_community.chat_message_histories import ChatMessageHistory

prompt_template = ChatPromptTemplate.from_messages([
    ("system","You are a helpful assistant . Answer the questions very shortly and briefly"),
    MessagesPlaceholder("history"),
    ("human","{input}")
])

chain = prompt_template | google_llm | StrOutputParser()

conversation = {}

def chat_with_bot(payload):
    input = payload["input"]
    session_id  = payload["session_id"]
    if session_id not in conversation:
        conversation[session_id] = ChatMessageHistory()
    history = conversation[session_id]
    history.add_user_message(input)
    res = chain.invoke({"input": input, "history":history.messages})
    history.add_ai_message(res)
    return res

print(conversation)
res1=chat_with_bot({"input":"Who is Joseph vijay?","session_id":"user1"})
print(conversation)
res2=chat_with_bot({"input":"What is his age?","session_id":"user2"})
print(conversation)

print(res1,"\n",res2)

{}


C:\Users\gokul\AppData\Local\Temp\ipykernel_28408\1499145488.py:20: LangChainDeprecationWarning: The class `InMemoryChatMessageHistory` was deprecated in LangChain 1.6.4 and will be removed in 2.0.0 See the short-term memory documentation for recommended alternatives: https://docs.langchain.com/oss/python/langchain/short-term-memory
  conversation[session_id] = ChatMessageHistory()


{'user1': InMemoryChatMessageHistory(messages=[HumanMessage(content='Who is Joseph vijay?', additional_kwargs={}, response_metadata={}), AIMessage(content='Joseph Vijay, known professionally as **Vijay**, is a prominent Indian actor and politician who primarily works in Tamil cinema.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])])}
{'user1': InMemoryChatMessageHistory(messages=[HumanMessage(content='Who is Joseph vijay?', additional_kwargs={}, response_metadata={}), AIMessage(content='Joseph Vijay, known professionally as **Vijay**, is a prominent Indian actor and politician who primarily works in Tamil cinema.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='What is his age?', additional_kwargs={}, response_metadata={}), AIMessage(content='He is 50 years old (born June 22, 1974).', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])])}
Joseph Vijay, known profe

RunnableWithMessageHistory

In [21]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.prompts import MessagesPlaceholder
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables import RunnableWithMessageHistory

prompt_template = ChatPromptTemplate.from_messages([
    ("system","You are a helpful assistant . Answer the questions very shortly and briefly"),
    MessagesPlaceholder("history"),
    ("human","{input}")
])

chain = prompt_template | google_llm | StrOutputParser()

store = {}

def get_session_history(session_id):
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

chain_with_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history"
)

print(conversation)
res1=chain_with_history.invoke({"input":"Who is Joseph vijay?"},config={"configurable": {"session_id":"user1"}})
print(conversation)
res2=chain_with_history.invoke({"input":"What is his age?"},config={"configurable": {"session_id":"user1"}})
print(conversation)

print(res1,"\n",res2)

c:\Users\gokul\OneDrive\Desktop\Langchain\venv\Lib\site-packages\IPython\core\interactiveshell.py:3823: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


{'user1': InMemoryChatMessageHistory(messages=[HumanMessage(content='Who is Joseph vijay?', additional_kwargs={}, response_metadata={}), AIMessage(content='Joseph Vijay, known professionally as **Vijay**, is a prominent Indian actor and politician who primarily works in Tamil cinema.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='What is his age?', additional_kwargs={}, response_metadata={}), AIMessage(content='He is 50 years old (born June 22, 1974).', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])])}
{'user1': InMemoryChatMessageHistory(messages=[HumanMessage(content='Who is Joseph vijay?', additional_kwargs={}, response_metadata={}), AIMessage(content='Joseph Vijay, known professionally as **Vijay**, is a prominent Indian actor and politician who primarily works in Tamil cinema.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='What 